# Convert Torch Model to Onnx

In [5]:
import yaml
import torch 

from animl.classifiers import EfficientNet

2024-04-29 09:28:51.314549: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-29 09:28:51.314599: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-29 09:28:51.334030: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-29 09:28:51.384057: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-29 09:28:52.189294: W tensorflow/comp

In [6]:
def load_model(cfg):
    '''
        Creates a model instance. 
    '''
    if (cfg['architecture']=="CTL"):
        model_instance = CTLClassifier(cfg['num_classes'])
    elif (cfg['architecture']=="efficientnet_v2_m"):
        model_instance = EfficientNet(cfg['num_classes'],tune=False)        
    else:
        raise AssertionError('Please provide the correct model')

    return model_instance

In [7]:
#location of the torch model
PATH = "/mnt/machinelearning/Models/Andes/Experiment2/30.pt"

#load the config with parameters
cfg = yaml.safe_load(open("/mnt/machinelearning/Models/Andes/Experiment2/exp2.yaml", 'r'))

image_size= cfg["image_size"]
num_epochs= cfg["num_epochs"]

batch_size = cfg["batch_size"]

In [8]:
#initialize model based on cfg's model
modelInstance = load_model(cfg)

/home/kyra/anaconda3/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_M_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [9]:
#load the torch model from the path
pathModelAll = torch.load(PATH, map_location=torch.device('cpu'))

#get the dict of the given model instance version of model from path 
modelInstance.load_state_dict(pathModelAll["model"])

<All keys matched successfully>

In [10]:
#get the model ready
modelInstance.eval()

EfficientNet(
  (avgpool): AdaptiveAvgPool2d(output_size=1)
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)

In [11]:
#changing from torch to onnx

# Input to the model
x = torch.randn(batch_size, 3, image_size[0], image_size[1], requires_grad=True)
torch_out = modelInstance(x)

# Export the model
torch.onnx.export(modelInstance,               # model being run
                  x,                         # model input (or a tuple for multiple inputs)
                  "onnxModel.onnx",          # where to save the model (can be a file or file-like object)
                  export_params=True,        # store the trained parameter weights inside the model file
                  opset_version=10,          # the ONNX version to export the model to
                  do_constant_folding=True,  # whether to execute constant folding for optimization
                  input_names = ['input'],   # the model's input names
                  output_names = ['output'], # the model's output names
                  dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                                'output' : {0 : 'batch_size'}})

In [14]:
torch_out.shape

torch.Size([32, 53])